In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/divjotbedi/titanic-without-pipeline/__results__.html
/kaggle/input/notebooks/divjotbedi/titanic-without-pipeline/__notebook__.ipynb
/kaggle/input/notebooks/divjotbedi/titanic-without-pipeline/__output__.json
/kaggle/input/notebooks/divjotbedi/titanic-without-pipeline/custom.css
/kaggle/input/notebooks/divjotbedi/titanic-without-pipeline/models/oe_sex.pkl
/kaggle/input/notebooks/divjotbedi/titanic-without-pipeline/models/clf.pkl
/kaggle/input/notebooks/divjotbedi/titanic-without-pipeline/models/oe_embarked.pkl
/kaggle/input/datasets/brendan45774/test-file/tested.csv


In [2]:
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import chi2,SelectKBest

In [3]:
df=pd.read_csv("/kaggle/input/datasets/brendan45774/test-file/tested.csv")

In [4]:
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
5,897,0,3,"Svensson, Mr. Johan Cervin",male,14.0,0,0,7538,9.2250,NaN,S
6,898,1,3,"Connolly, Miss. Kate",female,30.0,0,0,330972,7.6292,NaN,Q
7,899,0,2,"Caldwell, Mr. Albert Francis",male,26.0,1,1,248738,29.0000,NaN,S
8,900,1,3,"Abrahim, Mrs. Joseph (Sophie Halaut Easu)",female,18.0,0,0,2657,7.2292,NaN,C
9,901,0,3,"Davies, Mr. John Samuel",male,21.0,2,0,A/4 48871,24.1500,NaN,S


In [5]:
df.drop(columns=['Name','PassengerId','Ticket','Cabin'],inplace=True)

In [6]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=["Survived"]),
                                              df['Survived'],
                                              random_state=42,
                                              test_size=0.2)

In [7]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
336,2,male,32.0,0,0,13.0000,S
31,2,male,24.0,2,0,31.5000,S
84,2,male,NaN,0,0,10.7083,Q
287,1,male,24.0,1,0,82.2667,S
317,2,male,19.0,0,0,10.5000,S


In [8]:
X_train.isnull().sum()

Pclass       0
Sex          0
Age         72
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64

In [9]:
trf1=ColumnTransformer([
    ('imputed_age',SimpleImputer(),[2]),
    ('imputed_fare',SimpleImputer(strategy='most_frequent'),[5]),
],remainder = 'passthrough')

In [10]:
trf2=ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6]),
],remainder = 'passthrough')

In [11]:
trf3=ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10)),
])

In [12]:
trf4=SelectKBest(score_func=chi2,k=5)

In [13]:
trf5=DecisionTreeClassifier()

In [14]:
pipe=Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5),
])

In [15]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('imputed_age',
                                                  SimpleImputer(), [2]),
                                                 ('imputed_fare',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [5])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=5,
                             score_func=<function chi2 at 0x7ab19f479580>)),
                ('trf5', DecisionTreeClassifier())])

In [16]:
y_pred=pipe.predict(X_test)

In [17]:
from sklearn.metrics import accuracy_score

In [18]:
accuracy_score(y_test,y_pred)

0.5952380952380952

In [19]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('imputed_age', SimpleImputer(), [2]),
                                 ('imputed_fare',
                                  SimpleImputer(strategy='most_frequent'),
                                  [5])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'trf3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=5, score_func=<function chi2 at 0x7ab19f479580>),
 'trf5': DecisionTreeClassifier()}

In [20]:
pipe.named_steps['trf1']

ColumnTransformer(remainder='passthrough',
                  transformers=[('imputed_age', SimpleImputer(), [2]),
                                ('imputed_fare',
                                 SimpleImputer(strategy='most_frequent'),
                                 [5])])

In [21]:
pipe.named_steps['trf1'].transformers_

[('imputed_age', SimpleImputer(), [2]),
 ('imputed_fare', SimpleImputer(strategy='most_frequent'), [5]),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 1, 3, 4, 6])]

In [22]:
pipe.named_steps['trf1'].transformers_[0]

('imputed_age', SimpleImputer(), [2])

In [23]:
pipe.named_steps['trf1'].transformers_[0][1]

SimpleImputer()

In [24]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_

array([30.11576336])

In [25]:
pipe.named_steps['trf1'].transformers_[1][1].statistics_

array([7.75])

In [26]:
cross_val_score(pipe,X_train,y_train,cv=5,scoring='accuracy').mean()

np.float64(0.6437358661239257)

In [27]:
params={
    'trf5__max_depth':[1,2,3,4,5,None]
}

In [28]:
Grid=GridSearchCV(pipe,params,cv=5,scoring='accuracy')
Grid.fit(X_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('imputed_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('imputed_fare',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [5])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('trf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf4',
                                        SelectKBest(k=5,
                                                    score_func=<function chi2 at 0x7ab19f479580>)),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [29]:
Grid.best_score_

np.float64(0.6467209407507915)

In [30]:
Grid.best_params_

{'trf5__max_depth': 1}

In [31]:
import pickle
import os
os.makedirs('models', exist_ok=True)

In [32]:
pickle.dump(pipe,open('models/pipe.pkl','wb'))